# DA5401 A8: Ensemble Learning for Complex Regression Modeling on Bike Share Data

Importing the necessary libraries


In [185]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

## Part A: Data Preprocessing and Baseline

### 1. Data Loading and Feature Engineering

In [186]:
# Load the dataset
df=pd.read_csv('hour.csv')

In [187]:
print("Original dataset shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
df.head()

Original dataset shape: (17379, 17)

First 5 rows of the dataset:


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [188]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


Drop irrelevant columns

In [189]:
columns_to_drop = ['instant', 'dteday', 'casual', 'registered']
df_clean = df.drop(columns=columns_to_drop)

Convert categorical features using One-Hot Encoding

In [190]:
categorical_features = ['season', 'weathersit', 'mnth', 'hr', 'yr', 'weekday']
df_encoded = pd.get_dummies(df_clean, columns=categorical_features, drop_first=True)

In [191]:
df_encoded.shape

(17379, 54)

### 2. Train/Test Split

In [192]:
X = df_encoded.drop('cnt', axis=1)
y = df_encoded['cnt']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 3. Baseline Model (Single Regressor)

Train Decision Tree Regressor

In [193]:
dt_regressor = DecisionTreeRegressor(max_depth=6, random_state=42)
dt_regressor.fit(X_train, y_train)
y_pred_dt = dt_regressor.predict(X_test)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
print(f"RMSE of decision tree: {rmse_dt}")

RMSE of decision tree: 118.45551730357617


Train Linear Regression model

In [194]:
lr_regressor = LinearRegression()
lr_regressor.fit(X_train, y_train)
y_pred_lr = lr_regressor.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"RMSE of Linear regression: {rmse_lr}")

RMSE of Linear regression: 100.44594623557187


Since RMSE of Linear regression is less, hence linear regression been chosen as a basemodel

In [195]:
baseline_model = "Linear Regression"
baseline_rmse = rmse_lr

## Part B: Ensemble Techniques for Bias and Variance Reduction

### 1. Bagging (Variance Reduction)

Implement Bagging Regressor with Decision Tree as base estimator

In [196]:
bagging_regressor = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=6),
    n_estimators=50,
    random_state=42
)

bagging_regressor.fit(X_train, y_train)
y_pred_bagging = bagging_regressor.predict(X_test)
rmse_bagging = np.sqrt(mean_squared_error(y_test, y_pred_bagging))

Compare with single Decision Tree baseline

In [197]:
print(f"Single Decision Tree RMSE: {rmse_dt:.2f}")
print(f"Bagging Regressor RMSE: {rmse_bagging:.2f}")
print(f"Variance Reduction: {rmse_dt - rmse_bagging:.2f} RMSE improvement")

Single Decision Tree RMSE: 118.46
Bagging Regressor RMSE: 112.34
Variance Reduction: 6.11 RMSE improvement


- Bagging effectively reduced variance compared to single Decision Tree baseline
- The hypothesis that bagging primarily targets variance reduction is supported

### 2. Boosting (Bias Reduction):

Implement Gradient Boosting Regressor

In [198]:
gradient_boosting = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

gradient_boosting.fit(X_train, y_train)
y_pred_boosting = gradient_boosting.predict(X_test)
rmse_boosting = np.sqrt(mean_squared_error(y_test, y_pred_boosting))

Performance comparison

In [199]:
print(f"Single Decision Tree RMSE: {rmse_dt:.2f}")
print(f"Bagging Regressor RMSE: {rmse_bagging:.2f}")
print(f"Gradient Boosting RMSE: {rmse_boosting:.2f}")

Single Decision Tree RMSE: 118.46
Bagging Regressor RMSE: 112.34
Gradient Boosting RMSE: 55.43


**Discussion of bias reduction**

**Bias Reduction Analysis**

✅ **Boosting achieved significantly better results** than both single model and bagging ensemble  
✅ This strongly supports the hypothesis that boosting primarily targets **bias reduction**  
✅ **Massive improvement:** 63.03 RMSE reduction over single model  
✅ **Massive improvement:** 56.91 RMSE reduction over bagging  

---

**Explanation**

- Gradient Boosting's sequential approach reduces bias by focusing on residual errors  
- Each new tree in the ensemble learns from the mistakes of previous trees  
- This addresses the fundamental limitations (bias) in the initial weak learners  
- The dramatic RMSE reduction (55.43 vs 118.46) indicates the single Decision Tree had high bias  
- Boosting successfully captured complex patterns that bagging and single trees missed  
- This demonstrates boosting's superior capability for **bias reduction in complex regression tasks**

---

**Performance Improvements**

- **Boosting vs Single Model:** 53.2% improvement  
- **Boosting vs Bagging:** 50.66% improvement  

📉 **Overall:** Boosting reduces prediction error by approximately **53.2%** compared to baseline


## Part C: Stacking for Optimal Performance

### 1. Stacking Implementation

**Stacking Principle Explanation:**

Stacking (Stacked Generalization) combines multiple machine learning models through a meta-learning framework. The principle involves:
 
- **Level-0 (Base Learners)**: Diverse models make initial predictions on the training data
- **Level-1 (Meta-Learner)**: A higher-level model that learns from the base learners' predictions
- **Optimal Combination**: The meta-learner discovers the optimal way to weight and combine the base learners' predictions, effectively learning which models perform best on different types of patterns in the data
 
The meta-learner is trained on the predictions of base learners using cross-validation to prevent overfitting and learn robust combination rules.

Define Base Learners (Level-0)

In [200]:
base_learners = [
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('bagging', bagging_regressor),
    ('boosting', gradient_boosting)
]

Define Meta-Learner (Level-1)

In [201]:
meta_learner = Ridge(alpha=1.0)

# Implement Stacking Regressor
stacking_regressor = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)

Train the stacking model

In [202]:
stacking_regressor.fit(X_train, y_train)

StackingRegressor(cv=5,
                  estimators=[('knn', KNeighborsRegressor()),
                              ('bagging',
                               BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=6),
                                                n_estimators=50,
                                                random_state=42)),
                              ('boosting',
                               GradientBoostingRegressor(max_depth=6,
                                                         random_state=42))],
                  final_estimator=Ridge())

### 2. Final Evaluation

Calculate RMSE for Stacking Regressor

In [203]:
y_pred_stacking = stacking_regressor.predict(X_test)
rmse_stacking = np.sqrt(mean_squared_error(y_test, y_pred_stacking))
print(f"RMSE for the Stacking Regressor: {rmse_stacking}")

RMSE for the Stacking Regressor: 53.17384633995361


## Part D: Final Analysis

### 1. Comparative Table

In [204]:
results_comparison = {
    'Model': [
        'Baseline Single Model (Linear Regression)', 
        'Bagging Regressor',
        'Gradient Boosting Regressor',
        'Stacking Regressor'
    ],
    'RMSE': [
        rmse_lr,  # Linear Regression RMSE
        rmse_bagging,  # Bagging Regressor RMSE
        rmse_boosting, # Gradient Boosting RMSE
        rmse_stacking, # Stacking Regressor RMSE
    ],
    'Type': [
        'Single Model',
        'Ensemble (Variance Reduction)',
        'Ensemble (Bias Reduction)',
        'Ensemble (Meta-Learning)'
    ]
}

results_df = pd.DataFrame(results_comparison)
results_df_sorted = results_df.sort_values('RMSE')
results_df

,Model,RMSE,Type
0,Baseline Single Model (Linear Regression),100.445946,Single Model
1,Bagging Regressor,112.343037,Ensemble (Variance Reduction)
2,Gradient Boosting Regressor,55.425154,Ensemble (Bias Reduction)
3,Stacking Regressor,53.173846,Ensemble (Meta-Learning)


### 2. Conclusion

**2. Conclusion**

---

**Best-Performing Model**

Based on the RMSE comparison table, the **Stacking Regressor** achieved the best overall performance with an RMSE of **53.17**.  
It outperformed all other models, including:

- **Baseline Linear Regression:** RMSE = 100.45  
- **Bagging Regressor:** RMSE = 112.34  
- **Gradient Boosting Regressor:** RMSE = 55.43  

The Stacking Regressor provided the most accurate predictions of bike rentals, demonstrating the effectiveness of combining multiple models for complex regression problems.  

---

**Explanation: Why the Stacking Regressor Outperformed the Baseline**

The superior performance of the **Stacking Regressor** can be explained through the **bias–variance trade-off** and **model diversity** concepts:

1. **Bias–Variance Trade-Off:**  
   - **Bagging** reduces *variance* by training multiple models (like Decision Trees) on random subsets of data and averaging their predictions. This stabilizes predictions but does not always reduce bias.  
   - **Boosting** reduces *bias* by sequentially correcting the errors of previous models, leading to more accurate results.  
   - **Stacking** takes this a step further by learning how to **combine multiple models** that capture different aspects of the data — thus minimizing both bias and variance simultaneously.  

2. **Model Diversity:**  
   The Stacking Regressor uses three diverse base learners — **K-Nearest Neighbors**, **Bagging Regressor**, and **Gradient Boosting Regressor** — each with distinct learning mechanisms.  
   - KNN captures local patterns in the data.  
   - Bagging provides robustness by averaging over multiple models.  
   - Gradient Boosting captures complex non-linear relationships.  
   The **meta-learner (Ridge Regression)** then learns the optimal way to combine these base predictions, leveraging their complementary strengths.  

3. **Resulting Effect:**  
   This diverse and hierarchical learning structure allows stacking to generalize better on unseen data, achieving a lower RMSE than any individual model. It effectively balances **low bias** (from boosting) and **low variance** (from bagging and averaging effects), which is why it performs best overall.

---

**Final Statement:**  
The **Stacking Regressor** emerged as the most effective model because it integrates the strengths of multiple ensemble and non-ensemble methods. Through model diversity and an optimal bias–variance balance, stacking delivers superior predictive accuracy and robustness compared to a single model baseline.
